In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Extended PSD analysis: compare multiple subjects
base_path = "D:/Users/windows/Desktop/Inner Speech Decoding/Dataset/"
subject = 'sub-01'
session = 'ses-01'
file_path = f"{base_path}{subject}_{session}_eeg-epo.fif"
epochs = mne.read_epochs(file_path, preload=True)
print(epochs)
print("Event IDs:", epochs.event_id)

In [ ]:
label_map = {31: 'up', 32: 'down', 33: 'right', 34: 'left'}
classes = sorted(label_map.keys())
sfreq = epochs.info['sfreq']
ch_names = epochs.ch_names
n_channels = len(ch_names)

In [ ]:
# Compute PSD per class
psd_per_class = {}
for code in classes:
    label = label_map[code]
    ep = epochs[str(code)] if str(code) in epochs.event_id else epochs[label]
    spectrum = ep.compute_psd(method='welch', fmin=0.5, fmax=60.0, n_fft=256, verbose=False)
    psd_per_class[label] = spectrum.get_data()  # (n_epochs, n_channels, n_freqs)

freqs = spectrum.freqs

In [ ]:
# Topographic PSD difference: class vs. rest (approximated by other classes)
# For each channel, compute relative power in alpha (8-13 Hz) and beta (13-30 Hz)
alpha_mask = (freqs >= 8) & (freqs < 13)
beta_mask = (freqs >= 13) & (freqs < 30)

print(f"{'Channel':<12} {'Class':<8} {'Alpha power':>14} {'Beta power':>14}")
print('-' * 52)
for label, psd in psd_per_class.items():
    for ch_idx, ch in enumerate(ch_names):
        alpha_p = psd[:, ch_idx, alpha_mask].mean()
        beta_p = psd[:, ch_idx, beta_mask].mean()
        print(f"{ch:<12} {label:<8} {alpha_p:>14.3e} {beta_p:>14.3e}")

In [ ]:
# PSD difference between pairs of classes (sub-plot grid)
n_classes = len(classes)
fig, axes = plt.subplots(n_classes, n_classes, figsize=(14, 12), sharex=True)

for i, code_i in enumerate(classes):
    for j, code_j in enumerate(classes):
        ax = axes[i, j]
        label_i = label_map[code_i]
        label_j = label_map[code_j]
        if i == j:
            psd_mean = psd_per_class[label_i].mean(axis=(0, 1))
            ax.semilogy(freqs, psd_mean)
            ax.set_title(label_i, fontsize=8)
        else:
            diff = (psd_per_class[label_i].mean(axis=(0, 1)) -
                    psd_per_class[label_j].mean(axis=(0, 1)))
            ax.plot(freqs, diff)
            ax.axhline(0, color='k', lw=0.5)
            ax.set_title(f'{label_i} - {label_j}', fontsize=8)
        ax.set_xlim([0.5, 60])

plt.suptitle('PSD Comparison Grid (diagonal: mean PSD, off-diagonal: pairwise difference)')
plt.tight_layout()
plt.show()